# Image réelle vs image générée : détection par CNN et transfer learning

Projet ESILV « Neural Networks & Deep Learning ».

**Tâche** : classer une image en **image réelle** (photo prise par un appareil) ou **image générée** (entièrement produite par un modèle d'IA), quel que soit le générateur.

**Méthode** : ResNet50 pré-entraîné sur ImageNet, fine-tuné en deux phases sur le dataset `cartografia/unbiased-tiny-genimage` (GenImage débiaisé : tout en JPEG qualité 96).

**Deux expériences, même modèle** :
- **« tous générateurs »** : entraînement et test sur les 7 générateurs ;
- **« générateurs mis de côté »** : entraînement sans Midjourney ni BigGAN, puis test sur ces deux générateurs jamais vus.

**Entraînement n° 2** : par rapport au premier entraînement (`docs/rapport_train1.md`), on ajoute une **augmentation flou + JPEG aléatoire** sur les deux classes, pour que le modèle ne puisse plus utiliser la netteté comme indice, et un **arrêt anticipé** sur les deux phases, avec plus d'epochs.

Le notebook s'exécute tel quel sur Colab ou Kaggle, de haut en bas. Toutes les décisions sont justifiées dans `docs/cadrage.md`.

## 1. Installation des packages

Sur Colab ou Kaggle, on installe ce qui manque avec `pip`. En local, le projet est géré avec **uv** : il faut installer avec `uv add`, pour que `pyproject.toml` et `uv.lock` restent à jour.

In [ ]:
import importlib.util
import os
import subprocess
import sys

# Nom du module importé -> nom du paquet à installer
PAQUETS_REQUIS = {
    "tensorflow": "tensorflow",
    "kagglehub": "kagglehub",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
}

# Colab et Kaggle se reconnaissent à des variables d'environnement qu'ils définissent eux-mêmes
SUR_LE_CLOUD = "COLAB_RELEASE_TAG" in os.environ or "KAGGLE_KERNEL_RUN_TYPE" in os.environ

paquets_manquants = [paquet for module, paquet in PAQUETS_REQUIS.items() if importlib.util.find_spec(module) is None]

if not paquets_manquants:
    print("Tous les paquets sont déjà installés.")
elif SUR_LE_CLOUD:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *paquets_manquants])
    print("Installés :", paquets_manquants)
else:
    raise ImportError(
        f"Paquets manquants : {paquets_manquants}. En local, installe-les avec : uv add {' '.join(paquets_manquants)}"
    )

## 2. Imports

In [ ]:
import json
import random
from pathlib import Path

import kagglehub
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU disponibles :", tf.config.list_physical_devices("GPU"))

## 3. Téléchargements

Le dataset vient de Kaggle (`cartografia/unbiased-tiny-genimage`, 2,5 Go, licence CC BY-NC-SA 4.0). Sur Kaggle, il suffit de l'ajouter au notebook (« Add Input ») ; ailleurs, `kagglehub` le télécharge.

On télécharge aussi tout de suite les poids ImageNet de ResNet50, pour que tous les téléchargements soient faits en début de notebook.

`SMOKE_TEST = True` lance une version miniature (quelques images, 1 epoch, sans poids pré-entraînés) qui sert seulement à vérifier que le notebook tourne de bout en bout.

In [ ]:
# Mode test rapide : True pour vérifier que tout tourne, False pour le vrai entraînement
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"

KAGGLE_DATASET = "cartografia/unbiased-tiny-genimage"
KAGGLE_INPUT_DIR = Path("/kaggle/input/unbiased-tiny-genimage")
REAL_FOLDER = "Nature"  # dossier des images réelles dans le dataset

# Pas de poids pré-entraînés en mode test, pour éviter un téléchargement inutile
WEIGHTS = None if SMOKE_TEST else "imagenet"


def find_dataset_root(start_dir):
    """Cherche le dossier du dataset qui contient le sous-dossier des images réelles.

    Selon la plateforme, le dataset peut être rangé un ou deux niveaux plus bas
    que le dossier de téléchargement.

    Args:
        start_dir: dossier à partir duquel chercher.

    Returns:
        Le chemin (Path) du dossier qui contient `Nature/` et les dossiers des générateurs.
    """
    start_dir = Path(start_dir)
    for folder, subfolders, _ in os.walk(start_dir):
        if REAL_FOLDER in subfolders:
            return Path(folder)
    raise FileNotFoundError(f"Aucun dossier '{REAL_FOLDER}' trouvé sous {start_dir}")


# Priorité : chemin imposé par variable d'environnement, puis dataset attaché sur Kaggle, puis téléchargement
if os.environ.get("GENIMAGE_DIR"):
    download_dir = Path(os.environ["GENIMAGE_DIR"])
elif KAGGLE_INPUT_DIR.exists():
    download_dir = KAGGLE_INPUT_DIR
else:
    download_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET))

DATA_DIR = find_dataset_root(download_dir)
print("Dataset :", DATA_DIR)
print("Dossiers :", sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))

# Télécharge et met en cache les poids ImageNet de ResNet50 (environ 95 Mo)
if WEIGHTS == "imagenet":
    keras.applications.ResNet50(include_top=False, weights="imagenet")
    print("Poids ImageNet de ResNet50 en cache.")

## 4. Configuration, sous-ensembles équilibrés et split

Les deux expériences sont **équilibrées 50/50** : on garde toutes les images réelles et on tire le même nombre d'images dans chaque générateur, pour avoir autant d'images générées que de réelles.

| | « tous générateurs » | « générateurs mis de côté » |
|---|---|---|
| Générateurs à l'entraînement | les 7 | ADM, GLIDE, VQDM, SD 1.5, Wukong |
| Images générées par générateur | ~833 | ~1 165 |

Chaque sous-ensemble est découpé en **70 / 15 / 15** (train / validation / test), stratifié par classe et par générateur.

In [ ]:
SEED = 42

GENERATORS = ["ADM", "BigGAN", "Midjourney", "VQDM", "glide", "stable_diffusion_v_1_5", "wukong"]
HELD_OUT_GENERATORS = ["Midjourney", "BigGAN"]  # jamais vus à l'entraînement de l'expérience 2

CROP_SIZE = 128  # taille de BigGAN, la plus petite du dataset : crop possible partout sans redimensionner
INPUT_SIZE = 224  # taille d'entrée habituelle de ResNet50
BATCH_SIZE = 8 if SMOKE_TEST else 64
# Nombres d'epochs maximaux : l'arrêt anticipé coupe avant si la validation ne progresse plus
EPOCHS_PHASE1 = 1 if SMOKE_TEST else 10
EPOCHS_PHASE2 = 1 if SMOKE_TEST else 30
LR_PHASE1 = 1e-3
LR_PHASE2 = 1e-5
DROPOUT = 0.3
PATIENCE = 4  # arrêt anticipé : epochs sans amélioration de la perte de validation avant d'arrêter
THRESHOLD = 0.5  # probabilité au-delà de laquelle une image est classée « générée »
MAX_IMAGES_PER_FOLDER = 60 if SMOKE_TEST else None

# Augmentation à l'entraînement uniquement, appliquée aux deux classes (voir section 5)
AUGMENT = True
BLUR_PROBABILITY = 0.5
BLUR_SIGMA_MAX = 2.0  # σ du flou tiré au hasard entre 0 et cette valeur, en pixels
JPEG_PROBABILITY = 0.5
JPEG_QUALITY_MIN = 50  # qualité JPEG tirée au hasard entre ces deux bornes
JPEG_QUALITY_MAX = 100

RUN_NAME = "train2"  # préfixe des fichiers sauvegardés, pour ne pas confondre avec l'entraînement n° 1
# Le mode test rapide écrit dans un dossier à part : ses modèles miniatures ne peuvent pas être rechargés par erreur
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "outputs_smoke" if SMOKE_TEST else "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REUSE_SAVED_MODELS = True  # recharge un modèle déjà entraîné au lieu de le réentraîner


def set_seeds(seed):
    """Fixe toutes les graines aléatoires (Python, NumPy, TensorFlow) pour la reproductibilité.

    Args:
        seed: la graine à utiliser.
    """
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)


def list_images(data_dir, max_per_folder=None):
    """Recense toutes les images du dataset.

    Args:
        data_dir: dossier racine du dataset (contient `Nature/` et un dossier par générateur).
        max_per_folder: nombre maximal d'images gardées par dossier (None = toutes).

    Returns:
        DataFrame avec une ligne par image : `path`, `source` (nom du dossier) et
        `label` (0 = image réelle, 1 = image générée).
    """
    extensions = {".jpg", ".jpeg", ".png"}
    rows = []
    for source in [REAL_FOLDER, *GENERATORS]:
        paths = sorted(p for p in (data_dir / source).iterdir() if p.suffix.lower() in extensions)
        if max_per_folder is not None:
            paths = paths[:max_per_folder]
        label = 0 if source == REAL_FOLDER else 1
        rows += [{"path": str(p), "source": source, "label": label} for p in paths]
    return pd.DataFrame(rows)


def build_balanced_subset(inventory, generators, seed):
    """Construit un sous-ensemble équilibré 50/50 entre images réelles et générées.

    Toutes les images réelles sont gardées. Le même nombre d'images est tiré au
    hasard dans chaque générateur, de sorte que le total des images générées
    soit à peu près égal au nombre d'images réelles.

    Args:
        inventory: DataFrame renvoyé par `list_images`.
        generators: générateurs à inclure.
        seed: graine du tirage.

    Returns:
        Le sous-ensemble mélangé (DataFrame).
    """
    real = inventory[inventory["source"] == REAL_FOLDER]
    per_generator = len(real) // len(generators)
    generated = [
        inventory[inventory["source"] == g].sample(n=per_generator, random_state=seed)
        for g in generators
    ]
    return pd.concat([real, *generated]).sample(frac=1, random_state=seed).reset_index(drop=True)


def split_train_val_test(subset, seed):
    """Découpe un sous-ensemble en train / validation / test (70 / 15 / 15).

    Le split est stratifié sur `source`, donc à la fois par classe et par générateur.

    Args:
        subset: DataFrame à découper.
        seed: graine du découpage.

    Returns:
        Dictionnaire {"train": ..., "val": ..., "test": ...} de DataFrames.
    """
    train, rest = train_test_split(subset, test_size=0.30, stratify=subset["source"], random_state=seed)
    val, test = train_test_split(rest, test_size=0.50, stratify=rest["source"], random_state=seed)
    return {"train": train.reset_index(drop=True), "val": val.reset_index(drop=True), "test": test.reset_index(drop=True)}


def count_table(splits):
    """Compte les images par source et par partie du split, pour vérification.

    Args:
        splits: dictionnaire renvoyé par `split_train_val_test`.

    Returns:
        DataFrame (sources en lignes, parties en colonnes, avec un total).
    """
    table = pd.DataFrame({name: df["source"].value_counts() for name, df in splits.items()}).fillna(0).astype(int)
    table.loc["TOTAL"] = table.sum()
    return table


set_seeds(SEED)
inventory = list_images(DATA_DIR, MAX_IMAGES_PER_FOLDER)
print("Images trouvées par dossier :")
print(inventory["source"].value_counts().to_string())

# Expérience 1 : les 7 générateurs
splits_all = split_train_val_test(build_balanced_subset(inventory, GENERATORS, SEED), SEED)

# Expérience 2 : sans Midjourney ni BigGAN
TRAIN_GENERATORS_HELD_OUT = [g for g in GENERATORS if g not in HELD_OUT_GENERATORS]
splits_held_out = split_train_val_test(build_balanced_subset(inventory, TRAIN_GENERATORS_HELD_OUT, SEED), SEED)

# Test sur générateurs jamais vus : toutes leurs images, face aux images réelles du test de l'expérience 2
held_out_test = pd.concat([
    inventory[inventory["source"].isin(HELD_OUT_GENERATORS)],
    splits_held_out["test"][splits_held_out["test"]["label"] == 0],
]).reset_index(drop=True)

print("\nExpérience « tous générateurs » :")
print(count_table(splits_all))
print("\nExpérience « générateurs mis de côté » :")
print(count_table(splits_held_out))
print("\nTest sur générateurs mis de côté :")
print(held_out_test["source"].value_counts().to_string())

## 5. Prétraitement

Le piège de GenImage : chaque générateur a sa propre taille (128 à 1024 px) alors que les réelles font ~500 px. Si on redimensionne chaque image d'un facteur différent selon sa classe, le modèle apprend ces traces de redimensionnement au lieu d'apprendre « réel vs généré ».

La chaîne est donc **identique pour toutes les images** :
1. décodage en pixels RGB (les métadonnées disparaissent) ;
2. **crop de 128 × 128 px à résolution native**, aléatoire à l'entraînement et centré en validation et en test ;
3. **agrandissement à 224 × 224 px**, même facteur ×1,75 pour toutes les images ;
4. normalisation `resnet50.preprocess_input`.

**Augmentation (entraînement seulement, nouveauté de l'entraînement n° 2)** : avant le crop, à résolution native, chaque image d'entraînement, **réelle ou générée**, reçoit au hasard :
- un **flou gaussien** (probabilité 0,5, σ entre 0 et 2) ;
- une **recompression JPEG** (probabilité 0,5, qualité entre 50 et 100).

Le premier entraînement avait appris « net = réel » : environ 40 % des vraies photos floutées ou recompressées étaient classées « générées ». En dégradant les deux classes de la même façon, la netteté ne distingue plus les classes, et le modèle doit trouver des indices plus robustes. La validation et le test restent sans augmentation.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_image(path):
    """Lit un fichier image et le décode en tenseur RGB uint8 de forme (hauteur, largeur, 3).

    Args:
        path: chemin du fichier (tenseur chaîne).

    Returns:
        Tenseur uint8 de l'image.
    """
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])  # decode_image ne connaît pas la taille à l'avance
    return image


def crop_and_resize(image, training):
    """Découpe un carré de CROP_SIZE px à résolution native puis l'agrandit à INPUT_SIZE px.

    Args:
        image: tenseur uint8 (hauteur, largeur, 3).
        training: True pour un crop aléatoire, False pour un crop centré.

    Returns:
        Tenseur float32 (INPUT_SIZE, INPUT_SIZE, 3), valeurs entre 0 et 255.
    """
    if training:
        image = tf.image.random_crop(image, size=[CROP_SIZE, CROP_SIZE, 3])
    else:
        image = tf.image.resize_with_crop_or_pad(image, CROP_SIZE, CROP_SIZE)  # crop centré
    image = tf.cast(image, tf.float32)
    # Même facteur d'agrandissement (×1,75) pour toutes les images, quelle que soit leur classe
    return tf.image.resize(image, [INPUT_SIZE, INPUT_SIZE], method="bilinear")


def gaussian_blur(image, sigma, kernel_size=9):
    """Applique un flou gaussien.

    Args:
        image: tenseur uint8 (hauteur, largeur, 3).
        sigma: écart-type du noyau gaussien en pixels (nombre ou tenseur scalaire).
        kernel_size: taille du noyau (impair), assez grande pour σ <= kernel_size / 3.

    Returns:
        Tenseur uint8 flouté, de même taille.
    """
    sigma = tf.maximum(tf.cast(sigma, tf.float32), 1e-3)  # évite une division par zéro quand σ ≈ 0
    axis = tf.range(kernel_size, dtype=tf.float32) - (kernel_size - 1) / 2
    kernel_1d = tf.exp(-(axis**2) / (2 * sigma**2))
    kernel_2d = tf.tensordot(kernel_1d, kernel_1d, axes=0)
    kernel_2d /= tf.reduce_sum(kernel_2d)
    # Même noyau appliqué séparément à chacun des 3 canaux (convolution « depthwise »)
    kernel = tf.tile(kernel_2d[:, :, None, None], [1, 1, 3, 1])
    blurred = tf.nn.depthwise_conv2d(tf.cast(image, tf.float32)[None], kernel, strides=[1, 1, 1, 1], padding="SAME")[0]
    return tf.cast(tf.clip_by_value(tf.round(blurred), 0, 255), tf.uint8)


def random_degradation(image):
    """Dégrade aléatoirement une image d'entraînement : flou puis recompression JPEG, chacun au hasard.

    Appliquée de la même façon aux images réelles et générées, pour que la netteté
    ou la compression ne permettent plus de distinguer les classes.

    Args:
        image: tenseur uint8 (hauteur, largeur, 3), à résolution native.

    Returns:
        Tenseur uint8 éventuellement flouté et/ou recompressé.
    """
    apply_blur = tf.random.uniform([]) < BLUR_PROBABILITY
    sigma = tf.random.uniform([], 0.0, BLUR_SIGMA_MAX)
    image = tf.cond(apply_blur, lambda: gaussian_blur(image, sigma), lambda: image)

    apply_jpeg = tf.random.uniform([]) < JPEG_PROBABILITY
    image = tf.cond(
        apply_jpeg,
        lambda: tf.image.random_jpeg_quality(image, JPEG_QUALITY_MIN, JPEG_QUALITY_MAX),
        lambda: image,
    )
    image.set_shape([None, None, 3])  # tf.cond perd parfois la forme statique
    return image


def make_dataset(df, training, perturbation=None, batch_size=None):
    """Construit le pipeline tf.data d'un DataFrame d'images.

    Args:
        df: DataFrame avec les colonnes `path` et `label`.
        training: True pour mélanger, augmenter (si AUGMENT) et faire des crops aléatoires.
        perturbation: fonction optionnelle appliquée à l'image uint8 décodée
            (utilisée par le test de perturbation).
        batch_size: taille de batch (BATCH_SIZE par défaut).

    Returns:
        tf.data.Dataset de couples (image prétraitée, label).
    """
    dataset = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label"].values.astype("float32")))
    if training:
        dataset = dataset.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)

    def load(path, label):
        image = decode_image(path)
        if training and AUGMENT:
            image = random_degradation(image)  # avant le crop, à résolution native
        if perturbation is not None:
            image = perturbation(image)
        image = crop_and_resize(image, training)
        return keras.applications.resnet50.preprocess_input(image), label

    return dataset.map(load, num_parallel_calls=AUTOTUNE).batch(batch_size or BATCH_SIZE).prefetch(AUTOTUNE)


def load_display_image(path):
    """Charge une image comme le modèle la voit en évaluation (crop centré agrandi), pour l'afficher.

    Args:
        path: chemin du fichier image.

    Returns:
        Tableau NumPy uint8 (INPUT_SIZE, INPUT_SIZE, 3).
    """
    image = crop_and_resize(decode_image(tf.constant(path)), training=False)
    return tf.cast(image, tf.uint8).numpy()


def show_examples(inventory, per_source=2):
    """Affiche quelques images de chaque dossier, telles que le modèle les voit.

    Args:
        inventory: DataFrame renvoyé par `list_images`.
        per_source: nombre d'images par dossier.
    """
    sources = [REAL_FOLDER, *GENERATORS]
    fig, axes = plt.subplots(per_source, len(sources), figsize=(2.2 * len(sources), 2.4 * per_source))
    for col, source in enumerate(sources):
        sample = inventory[inventory["source"] == source].sample(per_source, random_state=SEED)
        for row, path in enumerate(sample["path"]):
            ax = axes[row, col]
            ax.imshow(load_display_image(path))
            ax.set_title(source if row == 0 else "", fontsize=8)
            ax.axis("off")
    fig.suptitle("Crops de 128 px agrandis en 224 px (réelles = Nature)")
    plt.tight_layout()
    plt.show()


show_examples(inventory)

## 6. Construction du modèle

- **Backbone** : ResNet50 pré-entraîné sur ImageNet, sans sa tête de classification.
- **Tête** : GlobalAveragePooling → Dropout(0,3) → 1 neurone sigmoïde (probabilité que l'image soit générée).
- **Phase 1** : backbone gelé, seule la tête apprend (lr 1e-3, 10 epochs au maximum).
- **Phase 2** : dégel du dernier bloc `conv5` (lr 1e-5, 30 epochs au maximum).
- **Arrêt anticipé sur les deux phases** : on arrête quand la perte de validation ne s'améliore plus pendant 4 epochs, et on revient aux poids de la meilleure epoch. C'est ce qui évite le surapprentissage malgré le nombre d'epochs plus élevé.

Les BatchNorm restent en mode inférence (`training=False`), comme le recommande Keras pour le fine-tuning.

In [ ]:
# Précision mixte : plus rapide sur GPU, inutile sur CPU
if tf.config.list_physical_devices("GPU"):
    keras.mixed_precision.set_global_policy("mixed_float16")
print("Politique de précision :", keras.mixed_precision.global_policy().name)


def build_model(weights):
    """Construit ResNet50 + tête de classification binaire, backbone gelé.

    Args:
        weights: "imagenet" pour les poids pré-entraînés, None pour des poids aléatoires.

    Returns:
        Le modèle Keras complet.
    """
    backbone = keras.applications.ResNet50(include_top=False, weights=weights, input_shape=(INPUT_SIZE, INPUT_SIZE, 3))
    backbone.trainable = False

    inputs = keras.Input(shape=(INPUT_SIZE, INPUT_SIZE, 3))
    x = backbone(inputs, training=False)  # BatchNorm en mode inférence, même après dégel
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT)(x)
    # Sortie en float32 pour la stabilité numérique en précision mixte
    outputs = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    return keras.Model(inputs, outputs, name="resnet50_reel_vs_genere")


def unfreeze_last_block(model):
    """Dégèle le dernier bloc (conv5) de ResNet50, en laissant les BatchNorm gelées.

    Args:
        model: modèle renvoyé par `build_model`.
    """
    backbone = model.get_layer("resnet50")
    backbone.trainable = True
    for layer in backbone.layers:
        layer.trainable = layer.name.startswith("conv5_") and not isinstance(layer, layers.BatchNormalization)


def compile_model(model, learning_rate):
    """Compile le modèle avec Adam, l'entropie croisée binaire, l'accuracy et l'AUC.

    Args:
        model: le modèle Keras.
        learning_rate: taux d'apprentissage.
    """
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")],
    )


def early_stopping():
    """Crée le callback d'arrêt anticipé utilisé dans les deux phases.

    Il surveille la perte de validation, arrête après PATIENCE epochs sans
    amélioration et restaure les poids de la meilleure epoch.

    Returns:
        Le callback Keras.
    """
    return keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1
    )


def train_or_load(name, splits):
    """Entraîne le modèle d'une expérience en deux phases, ou le recharge s'il existe déjà.

    Les deux phases utilisent l'arrêt anticipé. Le modèle et l'historique
    d'entraînement sont sauvegardés dans OUTPUT_DIR, pour pouvoir réévaluer sans
    réentraîner.

    Args:
        name: nom de l'expérience (sert de nom de fichier, préfixé par RUN_NAME).
        splits: dictionnaire renvoyé par `split_train_val_test`.

    Returns:
        Couple (modèle, historique) ; l'historique est un dict métrique -> liste par epoch.
    """
    model_path = OUTPUT_DIR / f"{RUN_NAME}_{name}.keras"
    history_path = OUTPUT_DIR / f"{RUN_NAME}_{name}_history.json"
    if REUSE_SAVED_MODELS and model_path.exists():
        print(f"Modèle {name} rechargé depuis {model_path}")
        history = json.loads(history_path.read_text()) if history_path.exists() else {}
        return keras.models.load_model(model_path), history

    set_seeds(SEED)
    train_ds = make_dataset(splits["train"], training=True)
    val_ds = make_dataset(splits["val"], training=False)
    model = build_model(WEIGHTS)

    print(f"--- {name} : phase 1 (backbone gelé) ---")
    compile_model(model, LR_PHASE1)
    phase1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE1, callbacks=[early_stopping()])

    print(f"--- {name} : phase 2 (dégel de conv5) ---")
    unfreeze_last_block(model)
    compile_model(model, LR_PHASE2)  # recompiler est obligatoire après un changement de `trainable`
    phase2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_PHASE2, callbacks=[early_stopping()])

    # Historique des deux phases mis bout à bout
    history = {k: [float(v) for v in phase1.history[k] + phase2.history.get(k, [])] for k in phase1.history}
    history["phase2_start"] = len(phase1.history["loss"])
    model.save(model_path)
    history_path.write_text(json.dumps(history))
    return model, history


def plot_history(history, title):
    """Trace la perte et l'AUC (entraînement et validation) au fil des epochs.

    Args:
        history: historique renvoyé par `train_or_load`.
        title: titre du graphique.
    """
    if not history:
        print("Pas d'historique disponible.")
        return
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    for ax, metric in zip(axes, ["loss", "auc"]):
        ax.plot(history[metric], label="train")
        ax.plot(history[f"val_{metric}"], label="validation")
        ax.axvline(history["phase2_start"] - 0.5, color="grey", linestyle="--", label="début phase 2")
        ax.set_xlabel("epoch")
        ax.set_title(metric)
        ax.legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


build_model(WEIGHTS).summary(show_trainable=True)

## 7. Expérience « tous générateurs » : entraînement

Entraînement sur un mélange équilibré des 7 générateurs. Question : quel score obtient-on quand tous les générateurs sont connus ?

In [ ]:
model_all, history_all = train_or_load("tous_generateurs", splits_all)
plot_history(history_all, "Expérience « tous générateurs »")

## 8. Expérience « générateurs mis de côté » : entraînement

Même modèle, entraîné **sans Midjourney ni BigGAN**. Question : le modèle reconnaît-il des images venant de générateurs qu'il n'a jamais vus ?

In [ ]:
model_held_out, history_held_out = train_or_load("generateurs_mis_de_cote", splits_held_out)
plot_history(history_held_out, "Expérience « générateurs mis de côté »")

## 9. Évaluation et tableau comparatif

Métriques : accuracy, F1, AUC ROC, matrice de confusion, et **taux de bonnes réponses par source** (par générateur et pour les réelles). Seuil de décision : 0,5.

In [ ]:
def predict_probabilities(model, df, perturbation=None):
    """Calcule la probabilité « image générée » pour chaque image d'un DataFrame.

    Args:
        model: modèle entraîné.
        df: DataFrame avec `path` et `label`.
        perturbation: perturbation optionnelle (voir `make_dataset`).

    Returns:
        Tableau NumPy des probabilités, dans l'ordre de `df`.
    """
    dataset = make_dataset(df, training=False, perturbation=perturbation)
    return model.predict(dataset, verbose=0).ravel()


def compute_metrics(labels, probabilities):
    """Calcule accuracy, F1 et AUC.

    Args:
        labels: vrais labels (0 = réelle, 1 = générée).
        probabilities: probabilités « générée » prédites.

    Returns:
        Dictionnaire des métriques.
    """
    predictions = (probabilities >= THRESHOLD).astype(int)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
        "auc": roc_auc_score(labels, probabilities),
    }


def per_source_accuracy(df, probabilities):
    """Taux de bonnes réponses par source (dossier).

    Pour les réelles, c'est la part classée « réelle » ; pour un générateur,
    c'est la part classée « générée » (son taux de détection).

    Args:
        df: DataFrame avec `source` et `label`.
        probabilities: probabilités « générée » prédites.

    Returns:
        Série : source -> taux de bonnes réponses.
    """
    correct = (probabilities >= THRESHOLD).astype(int) == df["label"].values
    return pd.Series(correct, index=df["source"].values).groupby(level=0).mean().round(3)


def plot_confusion(labels, probabilities, title, ax):
    """Trace la matrice de confusion.

    Args:
        labels: vrais labels.
        probabilities: probabilités « générée » prédites.
        title: titre du graphique.
        ax: axe matplotlib où tracer.
    """
    matrix = confusion_matrix(labels, (probabilities >= THRESHOLD).astype(int), labels=[0, 1])
    ConfusionMatrixDisplay(matrix, display_labels=["réelle", "générée"]).plot(ax=ax, colorbar=False)
    ax.set_title(title, fontsize=9)


# Les trois évaluations du protocole
evaluations = {
    "« tous générateurs » → test": (model_all, splits_all["test"]),
    "« générateurs mis de côté » → test": (model_held_out, splits_held_out["test"]),
    "« générateurs mis de côté » → Midjourney + BigGAN": (model_held_out, held_out_test),
}

results, per_source, probabilities_by_eval = [], {}, {}
for name, (model, df) in evaluations.items():
    probabilities = predict_probabilities(model, df)
    probabilities_by_eval[name] = probabilities
    results.append({"évaluation": name, "images": len(df), **compute_metrics(df["label"], probabilities)})
    per_source[name] = per_source_accuracy(df, probabilities)

fig, axes = plt.subplots(1, len(evaluations), figsize=(5 * len(evaluations), 4))
for ax, (name, (model, df)) in zip(axes, evaluations.items()):
    plot_confusion(df["label"], probabilities_by_eval[name], name, ax)
plt.tight_layout()
plt.show()

print("Tableau comparatif :")
display(pd.DataFrame(results).set_index("évaluation").round(3))
print("Taux de bonnes réponses par source (taux de détection pour les générateurs) :")
display(pd.DataFrame(per_source))

## 10. Contrôles anti-triche

Deux contrôles pour vérifier que le modèle n'a pas appris un raccourci (format, compression, taille) :

1. **Scores par source** (tableau ci-dessus) : un générateur détecté à 100 % quand les autres plafonnent à 60 % serait suspect.
2. **Test de perturbation** : on applique **la même** transformation aux images de test **des deux classes**. Si le modèle s'appuie sur la netteté ou la compression, les prédictions basculent toutes vers une classe. Un modèle sain voit ses deux taux bouger peu.

Comme l'entraînement n° 2 apprend avec du flou (σ ≤ 2) et du JPEG (qualité ≥ 50), réussir sur ces dégradations-là ne prouve pas grand-chose. On teste donc deux groupes :
- **dans la plage de l'entraînement** : JPEG qualité 75, flou σ = 1 (les mêmes qu'à l'entraînement n° 1, pour comparer) ;
- **plus fortes ou jamais vues** : JPEG qualité 40, flou σ = 2,5, et réduction ×0,5 puis agrandissement (ce que font les réseaux sociaux). C'est ce groupe qui montre si le modèle est vraiment devenu robuste.

In [ ]:
def jpeg_recompression(image, quality=75):
    """Recompresse une image en JPEG à la qualité donnée.

    Args:
        image: tenseur uint8 (hauteur, largeur, 3).
        quality: qualité JPEG (0-100).

    Returns:
        Tenseur uint8 recompressé.
    """
    return tf.image.adjust_jpeg_quality(image, quality)


def downscale_upscale(image, factor=0.5):
    """Réduit l'image puis la ramène à sa taille d'origine, comme un réseau social qui redimensionne.

    Args:
        image: tenseur uint8 (hauteur, largeur, 3).
        factor: facteur de réduction (0,5 = moitié de la taille).

    Returns:
        Tenseur uint8 de même taille, qui a perdu ses détails les plus fins.
    """
    size = tf.shape(image)[:2]
    small_size = tf.cast(tf.cast(size, tf.float32) * factor, tf.int32)
    small = tf.image.resize(tf.cast(image, tf.float32), small_size, method="bilinear")
    restored = tf.image.resize(small, size, method="bilinear")
    return tf.cast(tf.clip_by_value(tf.round(restored), 0, 255), tf.uint8)


def perturbation_test(model, df, perturbations):
    """Compare la part d'images classées « générées » par classe, avant et après perturbation.

    Args:
        model: modèle entraîné.
        df: jeu de test (DataFrame avec `path`, `label`).
        perturbations: dictionnaire nom -> fonction de perturbation.

    Returns:
        DataFrame : une ligne par condition, la part classée « générée » parmi
        les réelles et parmi les générées.
    """
    rows = []
    for name, perturbation in {"aucune": None, **perturbations}.items():
        predicted_generated = predict_probabilities(model, df, perturbation) >= THRESHOLD
        rows.append({
            "perturbation": name,
            "réelles classées « générée »": predicted_generated[df["label"].values == 0].mean(),
            "générées classées « générée »": predicted_generated[df["label"].values == 1].mean(),
        })
    return pd.DataFrame(rows).set_index("perturbation").round(3)


PERTURBATIONS = {
    # Dans la plage de l'augmentation d'entraînement (et identiques au test de l'entraînement n° 1)
    "JPEG qualité 75": lambda image: jpeg_recompression(image, 75),
    "flou gaussien σ=1": lambda image: gaussian_blur(image, 1.0),
    # Plus fortes que l'augmentation, ou jamais vues à l'entraînement
    "JPEG qualité 40 (plus fort)": lambda image: jpeg_recompression(image, 40),
    "flou gaussien σ=2,5 (plus fort)": lambda image: gaussian_blur(image, 2.5, kernel_size=15),
    "réduction ×0,5 puis agrandissement (jamais vu)": downscale_upscale,
}

for name, (model, df) in list(evaluations.items())[:2]:
    print(f"Test de perturbation, {name} :")
    display(perturbation_test(model, df, PERTURBATIONS))

## 11. Grad-CAM : où le modèle regarde-t-il ?

Grad-CAM colore les zones de l'image qui ont le plus pesé dans la décision. On l'applique au modèle de l'expérience « générateurs mis de côté », sur la dernière couche de convolution de ResNet50 (`conv5_block3_out`), avec 8 exemples : 2 réelles, 2 générées bien classées, 2 erreurs et 2 images de générateurs jamais vus.

In [ ]:
def gradcam_heatmap(model, image):
    """Calcule la carte Grad-CAM d'une image pour la classe prédite.

    Args:
        model: modèle entraîné (ResNet50 + tête).
        image: image prétraitée, tenseur (INPUT_SIZE, INPUT_SIZE, 3).

    Returns:
        Couple (carte de chaleur NumPy entre 0 et 1, probabilité « générée »).
    """
    backbone = model.get_layer("resnet50")
    head = model.layers[model.layers.index(backbone) + 1:]  # GAP, Dropout, Dense
    with tf.GradientTape() as tape:
        feature_maps = tf.cast(backbone(image[None], training=False), tf.float32)  # sortie de conv5_block3_out
        tape.watch(feature_maps)
        output = feature_maps
        for layer in head:
            output = layer(output, training=False)
        probability = output[0, 0]
        # On explique la classe prédite : « générée » si p >= seuil, sinon « réelle »
        score = probability if probability >= THRESHOLD else 1 - probability
    gradients = tape.gradient(score, feature_maps)
    channel_weights = tf.reduce_mean(gradients, axis=(0, 1, 2))  # importance de chaque canal
    heatmap = tf.nn.relu(tf.reduce_sum(feature_maps[0] * channel_weights, axis=-1))
    heatmap /= tf.reduce_max(heatmap) + 1e-8
    return heatmap.numpy(), float(probability)


def pick_gradcam_examples(test_df, test_probabilities, held_df, held_probabilities, seed):
    """Choisit les 8 exemples : 2 réelles, 2 générées bien classées, 2 erreurs, 2 jamais vues.

    Args:
        test_df: jeu de test de l'expérience « générateurs mis de côté ».
        test_probabilities: probabilités prédites sur ce jeu.
        held_df: jeu de test des générateurs mis de côté.
        held_probabilities: probabilités prédites sur ce jeu.
        seed: graine du tirage.

    Returns:
        Liste de couples (titre, chemin de l'image).
    """
    test = test_df.assign(correct=(test_probabilities >= THRESHOLD).astype(int) == test_df["label"].values)
    held = held_df.assign(correct=(held_probabilities >= THRESHOLD).astype(int) == held_df["label"].values)
    groups = [
        ("réelle", test[(test["label"] == 0) & test["correct"]]),
        ("générée", test[(test["label"] == 1) & test["correct"]]),
        ("erreur", test[~test["correct"]]),
        ("jamais vue", held[held["source"].isin(HELD_OUT_GENERATORS)]),
    ]
    examples = []
    for kind, group in groups:
        for _, row in group.sample(min(2, len(group)), random_state=seed).iterrows():
            examples.append((f"{kind} ({row['source']})", row["path"]))
    return examples


def show_gradcam(model, examples):
    """Affiche chaque exemple avec sa carte Grad-CAM superposée.

    Args:
        model: modèle entraîné.
        examples: liste de couples (titre, chemin de l'image).
    """
    fig, axes = plt.subplots(2, len(examples), figsize=(2.4 * len(examples), 5))
    for col, (title, path) in enumerate(examples):
        display_image = load_display_image(path)
        model_input = keras.applications.resnet50.preprocess_input(display_image.astype("float32"))
        heatmap, probability = gradcam_heatmap(model, tf.constant(model_input))
        heatmap = tf.image.resize(heatmap[..., None], [INPUT_SIZE, INPUT_SIZE]).numpy()[..., 0]  # 7×7 -> 224×224
        axes[0, col].imshow(display_image)
        axes[0, col].set_title(f"{title}\np(générée)={probability:.2f}", fontsize=7)
        axes[1, col].imshow(display_image)
        axes[1, col].imshow(heatmap, cmap="jet", alpha=0.45)
        for row in range(2):
            axes[row, col].axis("off")
    fig.suptitle("Grad-CAM, expérience « générateurs mis de côté » (conv5_block3_out)")
    plt.tight_layout()
    plt.show()


name_test = "« générateurs mis de côté » → test"
name_held = "« générateurs mis de côté » → Midjourney + BigGAN"
gradcam_examples = pick_gradcam_examples(
    splits_held_out["test"], probabilities_by_eval[name_test],
    held_out_test, probabilities_by_eval[name_held], SEED,
)
show_gradcam(model_held_out, gradcam_examples)

## 12. Conclusion et limites

*À rédiger une fois les résultats obtenus.*

Points à commenter :
- **Résultats** : score de l'expérience « tous générateurs » contre l'expérience « générateurs mis de côté », et ce qu'on perd sur Midjourney et BigGAN, jamais vus.
- **Contrôles anti-triche** : les scores par source et le test de perturbation montrent-ils un raccourci ?
- **Grad-CAM** : le modèle regarde-t-il des zones cohérentes (textures, détails) ou des artefacts ?
- **Limites** : contenu limité aux catégories d'ImageNet (surtout animaux et objets) ; crop de 128 px, donc une petite partie des grandes images (Midjourney) ; un seul entraînement par expérience, sans recherche d'hyperparamètres.
- **Comparaison avec l'entraînement n° 1** (`docs/rapport_train1.md`) : ce que l'augmentation flou/JPEG coûte sur les images propres, et ce qu'elle apporte au test de perturbation.
- **Pistes** : plusieurs crops par image en évaluation (pour Midjourney), autres backbones (EfficientNet, ConvNeXt), jeu de test au contenu plus varié (Extended Synthbuster).